# Bài 23 — chuỗi luyện Easy · LeetCode #118 — Pascal's Triangle

**Mức độ:** Easy · **Chủ đề:** Dynamic Programming · 2D List · **Python:** 3  
**Đề gốc:** [https://leetcode.com/problems/pascals-triangle/](https://leetcode.com/problems/pascals-triangle/)

> Mục tiêu: hiểu cách tự suy luận từ ví dụ → viết code → trace trạng thái → chứng minh đúng → tự làm bài biến thể. Đọc Markdown, dự đoán rồi mới chạy cell code tiếp theo.

**Lưu ý:** đây là notebook dành cho người mới. Các cell TODO chỉ định nghĩa hàm, không được tự gọi khi chưa giải.

**Lưu ý học:** các ô TODO không được tự chạy; hãy tự làm, mở gợi ý từng mức rồi mới xem đáp án ở cuối.


**Lộ trình đầy đủ:** đề → nền tảng → baseline → điểm mấu chốt → code → trace → chứng minh → nộp LeetCode → Big-O → lỗi → biến thể → 6 bài tập → gợi ý → đáp án → checklist.

## 1. Đọc đề và xác định đúng đầu vào/đầu ra

Cho số hàng `numRows`, trả về `numRows` hàng đầu của tam giác Pascal. Hai đầu mỗi hàng bằng 1; mỗi số giữa bằng **tổng của hai số ngay phía trên**.

### Ví dụ quan trọng

`numRows=5 → [[1],[1,1],[1,2,1],[1,3,3,1],[1,4,6,4,1]]`; `numRows=1 → [[1]]`.

Hãy tự trả lời: **đầu ra là số, boolean, danh sách, hay node?** Với bài cấu trúc dữ liệu, phải chú ý **node/tham chiếu** khác **giá trị số**.

## 2. Thiết lập môi trường để chạy notebook

Trong Jupyter, hãy chạy cell tạo lớp dữ liệu và hàm hỗ trợ trước các cell sau. **Lưu ý:** LeetCode có sẵn một số lớp hoặc API; khi submit chỉ sao chép riêng cell `class Solution`/lớp thiết kế, không sao chép hàm giả lập dùng cho notebook.

In [ ]:
from __future__ import annotations
from typing import List, Optional
from collections import deque, Counter

### 2.1. `list` là gì? Thay đổi từng phần tử bằng chỉ số

Trong Python, `list` là dãy có thứ tự. Ví dụ `[10,20,30]` chứa ba phần tử ở chỉ số
`0`, `1`, `2`. Truy cập `a[i]` để đọc, gán `a[i] = value` để sửa phần tử tại vị trí i.
List là **mutable** (có thể thay đổi sau khi tạo). Vì vậy thuật toán tối ưu bộ nhớ có thể
tái sử dụng list hiện tại; nhưng sửa list quá sớm cũng có thể phá dữ liệu còn cần dùng.

`len(a)` trả số phần tử, còn chỉ số cuối là `len(a)-1`. Đọc `a[len(a)]` gây `IndexError`.
Hãy dự đoán giá trị cuối của list trước khi chạy thí nghiệm.

In [ ]:
list_demo=[10,20,30]
print('ban đầu:',list_demo,'độ dài:',len(list_demo))
print('vị trí 0:',list_demo[0],'vị trí cuối:',list_demo[-1])
list_demo[1]=99
print('sửa phần tử thứ hai:',list_demo)
assert list_demo==[10,99,30]

### 2.2. `for`, `range` và vì sao ranh giới phải chính xác

`for i in range(n)` duyệt các số nguyên `0,1,...,n-1`: *n không nằm trong range*.
`range(start, stop, step)` điều chỉnh điểm bắt đầu, điểm dừng và bước nhảy.
Nếu `step` âm, vòng lặp đi từ phải qua trái; đây là công cụ thiết yếu cho cập nhật
*in-place* (đặc biệt quan trọng ở Pascal II).

Luôn viết thử 3–4 giá trị chỉ số bằng tay trước khi thêm `a[i+1]` hoặc `a[i-1]`.
Việc chọn sai biên có thể gây truy cập ngoài mảng hoặc bỏ sót phần tử cuối.

In [ ]:
print('range(4):', list(range(4)))
print('range(1,4):', list(range(1,4)))
print('range(3,0,-1):', list(range(3,0,-1)))
a_indices=[5,8,13,21]
for i in range(1,len(a_indices)):
    print('cặp liên tiếp:',a_indices[i-1],a_indices[i])

### 2.3. `append`, phép cộng list và sao chép an toàn

`a.append(x)` thêm phần tử `x` vào **chính list a** (không tạo list mới, trả về `None`).
`a + b` tạo một list mới nối các phần tử, không sửa hai list ban đầu.
`b = a` chỉ tạo **bí danh** của a; muốn sao chép list phẳng dùng `a.copy()` hoặc `a[:]`.

Bẫy thường gặp: nếu lưu các hàng Pascal bằng cùng một list rồi sửa, các hàng cũ cũng đổi theo.
Với mảng giá cổ phiếu, hãy phân biệt rõ *thay đổi dữ liệu* với *đọc giá trong ngày*.
Thí nghiệm dưới đây sẽ chỉ ra sai lầm do aliasing một cách trực quan.

In [ ]:
original_list=[1,2,1]
alias_list=original_list
copy_list=original_list.copy()
original_list.append(1)
print('original:',original_list,'alias:',alias_list,'copy:',copy_list)
print('append trả:',copy_list.append(2))
print('copy đã thay đổi:',copy_list)
assert len(alias_list)==4 and len(copy_list)==4

### 2.4. `enumerate` cho cả vị trí và giá trị

Nếu cần biết phần tử đang ở **vị trí** nào (ngày thứ mấy, hàng thứ mấy),
`enumerate(items)` tạo lần lượt cặp `(index, value)`.
Nếu chỉ cần giá trị, `for value in items` gọn hơn.
`list(enumerate(...))` giúp quan sát trực tiếp các cặp chỉ số/giá trị.

Lưu ý `0-indexed`: ngày đầu hoặc hàng đầu có số thứ tự **0** trong Python.
Đây là nguyên nhân rất thường gặp của lỗi *off-by-one*: lệch đúng một vị trí.

In [ ]:
example_values=[7,1,5,3]
print('các cặp:',list(enumerate(example_values)))
for pos,value in enumerate(example_values):
    print(f'vị trí {pos} giữ giá trị {value}')
assert list(enumerate([8]))==[(0,8)]

### 2.5. Gọi hàm `min/max`, không nhầm với chuỗi phép so sánh

`min(a,b)` trả giá trị nhỏ hơn; `max(a,b)` trả giá trị lớn hơn.
Biểu thức `best=max(best,candidate)` là quy tắc phổ biến khi lưu **đáp án tốt nhất đến thời điểm hiện tại**.
Bạn có thể viết lại tương đương thành `if candidate > best: best = candidate`.

Hãy chú ý **best** giữ thông tin lịch sử, không nhất thiết bằng giá trị lớn nhất của chính input.
Ở bài Pascal, `max` không dùng để chọn số từ hàng cũ; số giữa được tính **bằng tổng**.
Cùng cú pháp Python, nhưng ý nghĩa toán học quyết định thuật toán.

In [ ]:
best_demo=0
for candidate_demo in (2,-3,5,4):
    old_best=best_demo
    best_demo=max(best_demo,candidate_demo)
    print('thử',candidate_demo,':',old_best,'->',best_demo)
assert best_demo==5

### 2.6. Định nghĩa Big-O bằng một ví dụ số nhỏ

**Time Complexity** nói số thao tác tăng thế nào khi input dài hơn.
Duyệt một vòng qua n phần tử thường tốn `O(n)`; hai vòng lồng nhau mỗi vòng chạy n lần
có thể tốn `O(n²)`. Tuy vậy phải đếm **số lần chạy thực** chứ không chỉ nhìn số vòng for:
nếu vòng trong chạy độ dài 1,2,3,...,n thì tổng là `n(n+1)/2 = O(n²)`.

**Space Complexity** tính dữ liệu phụ; nếu đầu ra yêu cầu chứa n phần tử thì riêng output đã cần `O(n)`.
Trong Pascal I trả toàn bộ tam giác cần `O(n²)` cho chính output, Pascal II chỉ trả một hàng `O(n)`.
Mục tiêu học là biết *n* đại diện cho cái gì trong từng bài.

In [ ]:
for n in (3,5,10):
    triangle_cells=n*(n+1)//2
    print(f'n={n}: số ô của n hàng đầu = {triangle_cells}; n² = {n*n}')
assert sum(range(1,6))==15

### 2.7. Quan sát trạng thái trước và sau khi cập nhật

Muốn hiểu một thuật toán theo vòng lặp, cần trả lời ba câu:
(1) Trước vòng lặp ta đã biết gì? (2) Ở bước này thay đổi biến nào?
(3) Sau bước này thông tin nào tiếp tục đúng?
Ta gọi thông tin được giữ đúng ấy là **invariant** (bất biến vòng lặp).

Để tránh bị rối, in `old_state` **trước** lệnh sửa và `new_state` **sau** lệnh sửa.
Nếu in mỗi đáp án cuối, ta không tìm được nguyên nhân lỗi ở giữa vòng lặp.
Bài minh họa dưới chỉ kiểm tra thao tác giữ tổng chạy, chưa sử dụng thuật toán của bài chính.

In [ ]:
running_sum=0
for number in [2,4,-1]:
    before=running_sum
    running_sum += number
    print(f'đọc {number:>2}: trước={before:>2}; sau={running_sum:>2}')
assert running_sum==5

## 3. Python nền tảng — công cụ cần hiểu trước khi giải

Danh sách lồng nhau `rows = [[1],[1,1]]`. `rows[-1]` là hàng vừa tạo, `rows[-1][j]` là phần tử thứ j của hàng đó. Chú ý `append()` thay đổi chính danh sách.

**Bài thí nghiệm:** thử thay dữ liệu trong cell dưới; các lệnh `print()` dùng để thấy giá trị trung gian, không phải mã submit.

In [ ]:
rows=[[1],[1,1],[1,2,1]]
print('hàng cuối:',rows[-1])
print('2 phần tử cha:',rows[-1][0],rows[-1][1])
print('tổng tạo ô giữa hàng sau:',rows[-1][0]+rows[-1][1])

### 3.1. Nhìn tam giác Pascal bằng hình và đọc chỉ số hàng/cột

Tam giác Pascal (Pascal’s Triangle) bắt đầu:

```text
            [1]                  hàng 0
           [1,1]                 hàng 1
          [1,2,1]                hàng 2
         [1,3,3,1]               hàng 3
        [1,4,6,4,1]              hàng 4
```

Hai mép của mỗi hàng luôn là 1. Ở giữa, phần tử tại cột `j` của hàng `r`
bằng **tổng hai phần tử ở hàng trước**, tại cột `j-1` và `j`.
Ký hiệu `P[r][j]`: `P[r][j]=P[r-1][j-1]+P[r-1][j]` với `1≤j<r`.
Hàng r có chính xác r+1 phần tử vì hai mép cộng r-1 số ở giữa.

Bài #118 yêu cầu **trả tất cả `numRows` hàng đầu**, không phải chỉ một hàng cuối.

In [ ]:
manual_triangle118=[[1],[1,1],[1,2,1],[1,3,3,1],[1,4,6,4,1]]
for row_index,row in enumerate(manual_triangle118):
    print(f'hàng {row_index}: {row}; có {len(row)} phần tử')
    assert len(row)==row_index+1
assert manual_triangle118[4][2]==manual_triangle118[3][1]+manual_triangle118[3][2]

### Tự kiểm tra ý nghĩa từng dòng

- Đầu vào của mỗi biểu thức là gì?
- Dòng nào tạo cấu trúc mới và dòng nào thay đổi cấu trúc hiện có?
- `True`/`False` và `None` có nghĩa gì trong trường hợp này?
- Nếu dữ liệu rỗng hoặc chỉ có một phần tử, đoạn mã có còn chạy được không?

## 4. Cách nghĩ trực giác (baseline)

Có thể tính từng phần tử hàng `r` bằng tổ hợp `C(r,k)` (n! / k!(n-k)!). Tuy đúng, công thức yêu cầu thêm giai thừa, không tận dụng kết quả hàng ngay trước.

**Hãy dự đoán** kết quả của hàm baseline trên các ví dụ trước khi chạy.

In [ ]:
def baseline(numRows):
    from math import comb
    return [[comb(r,c) for c in range(r+1)] for r in range(numRows)]

In [ ]:
for n in [1,3,5]:print(n,'->',baseline(n))

### 4.1. Baseline dùng `math.comb`: đúng toán học nhưng không giải thích cách hình thành

Công thức tổ hợp: số ở hàng n, cột k là `C(n,k)` (số cách chọn k phần tử từ n phần tử).
Trong Python, `math.comb(n,k)` tính số nguyên tổ hợp chính xác; ví dụ C(4,2)=6.
Bản baseline gọi công thức ấy cho từng vị trí và tạo cả tam giác.

Tuy nhiên, khi muốn **dạy tư duy DP**, ta cần hiểu cách hàng mới kế thừa hàng trước.
Nếu chỉ gọi một hàm thư viện cho mọi ô, người học có đáp án nhưng khó tự suy ra quy tắc cập nhật.
Chính vì vậy giải pháp chính xây dựng từng hàng từ hai ô ở hàng trước.
Không kết luận công thức tổ hợp là “sai” hay luôn chậm hơn: đây là hai cách nhìn có mục đích khác nhau.

In [ ]:
from math import comb
for n in range(5):
    print(n,[comb(n,k) for k in range(n+1)])
assert comb(4,2)==6
assert baseline(5)==[[1],[1,1],[1,2,1],[1,3,3,1],[1,4,6,4,1]]

### 4.2. Từ hàng trước đến hàng sau: tính bằng tay từng ô

Cho hàng trước `[1,3,3,1]`, ta muốn tạo hàng tiếp theo gồm 5 số.
Hai biên đặt `1` trước: `[1,?, ?, ?,1]`.
Cột 1 lấy `1+3=4`; cột 2 lấy `3+3=6`; cột 3 lấy `3+1=4`.
Kết quả `[1,4,6,4,1]`.

Đừng nhầm tính từ **hai phần tử kề nhau của cùng hàng mới** — đó là sai quan hệ truy hồi.
Mỗi ô ở giữa phải đọc từ một **hàng đã hoàn chỉnh trước đó**.
Một thuật toán dùng dữ liệu của bài toán nhỏ đã giải để xây bài toán lớn hơn là dạng
**Dynamic Programming (quy hoạch động)**, dù ở bài này rất trực quan.

In [ ]:
prev118=[1,3,3,1]
next118=[1]*(len(prev118)+1)
for j in range(1,len(next118)-1):
    next118[j]=prev118[j-1]+prev118[j]
    print(f'j={j}: {prev118[j-1]} + {prev118[j]} = {next118[j]}')
print('hàng mới:',next118)
assert next118==[1,4,6,4,1]

## 5. Vì sao cần chuyển sang kỹ thuật chính?

Đây là **Dynamic Programming**: một hàng được xây từ kết quả **hàng trước**. Ta không cần tính lại giai thừa.

Một thuật toán tốt cần trả lời: **ta đang nhớ thông tin gì, cập nhật khi nào, và làm sao không bỏ sót trường hợp nào?**

## 6. Lời giải chính — hiểu từng bước rồi mới viết

Bắt đầu `rows=[]`. Với mỗi độ dài `length=1..numRows`, tạo `row=[1]*length`; với những vị trí giữa `j=1..length-2`, gán `row[j]=prev[j-1]+prev[j]`.

Dưới đây là **hàm giải độc lập**. Khi submit, ta đưa cùng logic vào đúng chữ ký LeetCode.

### 5.1. Bất biến DP: các hàng trong `rows` đều hoàn chỉnh

Đặt `rows` là danh sách những hàng đã dựng được.
Trước mỗi lượt lặp, các hàng cũ phải đúng và **không bị sửa**.
Ta tạo `row` mới với chiều dài cần thiết, đặt hai mép là 1;
ô giữa đọc từ `prev=rows[-1]` (hàng cũ cuối cùng).
Sau khi mọi ô của `row` đúng, mới `rows.append(row)`.

**Bất biến:** sau khi hoàn thành hàng r, `rows` chứa đúng các hàng 0...r.
Base case hàng 0 là `[1]`; bước truy hồi chỉ dùng các giá trị đã xác định ở hàng r−1.
Vậy thuật toán đúng bằng quy nạp theo số hàng đã dựng.

In [ ]:
rows_state118=[]
for length in range(1,5):
    row=[1]*length
    if rows_state118:
        prev=rows_state118[-1]
        for j in range(1,length-1):row[j]=prev[j-1]+prev[j]
    rows_state118.append(row)
    print('đã hoàn chỉnh:',len(rows_state118),'hàng:',rows_state118)
assert rows_state118==baseline(4)

In [ ]:
def solve(numRows: int) -> list[list[int]]:
    rows=[]
    for length in range(1,numRows+1):
        row=[1]*length
        if rows:
            prev=rows[-1]
            for j in range(1,length-1):
                row[j]=prev[j-1]+prev[j]
        rows.append(row)
    return rows

### 6.1. Đọc từng dòng `row = [1] * length` và vòng lặp các ô giữa

`[1] * length` tạo một list có `length` số 1.
Nếu `length=1`, hàng đầu tiên là `[1]`, **không có ô giữa**.
Nếu `length=2`, hàng thứ hai `[1,1]`, cũng không có ô giữa.
Với `length=5`, chỉ cần sửa các chỉ số `1,2,3`;
`range(1,length-1)` tạo đúng `1,2,3` vì điểm dừng không được tính.

Ta có thể giữ nguyên code gọn mà vẫn hiểu từng phần: đầu tiên đặt 1 cho mọi ô,
rồi chỉ tính lại những ô nội bộ bằng hai ô hàng trước. Điều kiện `if rows:`
ngăn truy cập `rows[-1]` khi chưa có hàng nào.

In [ ]:
for length in (1,2,5):
    indexes=list(range(1,length-1))
    print('length',length,'hàng khởi tạo',[1]*length,'chỉ số cần cập nhật',indexes)
assert list(range(1,4))==[1,2,3]

### 6.2. Vì sao không được dùng một danh sách duy nhất làm mọi hàng?

Với bài #118, đầu ra cần **nhiều hàng độc lập**.
Nếu cùng một list được append nhiều lần vào `rows`, sửa lần sau sẽ sửa cả những hàng cũ.
Đây là lỗi *aliasing* ở list lồng nhau.

Bản code đúng tạo một list mới mỗi lượt (`row=[1]*length`) rồi append.
Ví dụ lỗi `rows=[old]*3`: cả ba mục chỉ đến cùng list `old`, không phải ba hàng riêng biệt.
Hãy so danh sách `bad` và `good` sau khi sửa một phần tử;
đây là thí nghiệm Python quan trọng trước khi học mảng hai chiều hoặc ma trận DP.

In [ ]:
old118=[1,1]
bad118=[old118]*3
good118=[old118.copy() for _ in range(3)]
bad118[0][0]=99
good118[0][0]=99
print('aliasing sai:',bad118)
print('copy độc lập:',good118)
assert bad118==[[99,1],[99,1],[99,1]]
assert good118==[[99,1],[1,1],[1,1]]

## 7. Trace bằng tay — quan sát trạng thái, không chỉ đáp án

| Hàng mới (dài) | Hàng trước | Tính ô giữa | Kết quả |
|---:|---|---|---|
| 1 | — | — | `[1]` |
| 2 | `[1]` | — | `[1,1]` |
| 3 | `[1,1]` | `1+1` | `[1,2,1]` |
| 4 | `[1,2,1]` | `1+2; 2+1` | `[1,3,3,1]` |

Đọc từ trái sang phải: mỗi dòng ghi **trạng thái trước khi quyết định**, hành động và phần thay đổi **sau khi quyết định**.

### 7.1. Bảng trace từng ô của hàng thứ năm (r=4)

Ta đã có bốn hàng 0..3; hàng 3 là `[1,3,3,1]`.

| j | `prev[j-1]` | `prev[j]` | `row[j]` sau | Ý nghĩa |
|---:|---:|---:|---:|---|
| 0 | — | — | 1 | Biên trái |
| 1 | 1 | 3 | 4 | 1+3 |
| 2 | 3 | 3 | 6 | 3+3 |
| 3 | 3 | 1 | 4 | 3+1 |
| 4 | — | — | 1 | Biên phải |

Đừng đọc bảng như phép cộng **trên hàng đang xây**: hai cột prev thuộc hàng trước,
vốn không bị sửa trong lúc xây hàng mới. Sau khi đủ năm phần tử, `rows.append(row)`.
Khi n=5, tổng số phần tử của các hàng phải là `1+2+3+4+5=15`.

In [ ]:
rows118=solve(5)
for r,row in enumerate(rows118):
    print(f'r={r}: {row} (len={len(row)})')
print('tổng phần tử:',sum(map(len,rows118)))
assert rows118[4]==[1,4,6,4,1]
assert sum(map(len,rows118))==15

### 7.2. Trace bảng trạng thái: `rows` trước / sau mỗi lần append

Bảng trace tốt cho người mới không chỉ ghi ra hàng cuối,
mà cho thấy hai trạng thái quan trọng:
(1) các hàng đã hoàn chỉnh **trước** lượt này;
(2) hàng mới được xây; (3) kết quả **sau** `append`.

`rows[-1]` là hàng cuối trong những hàng **đã được lưu**. Vì vậy, tại thời điểm tạo hàng r,
`rows[-1]` là hàng r−1. Nếu bạn append quá sớm trước khi tính ô giữa,
`rows[-1]` sẽ trỏ vào **chính hàng đang xây**, làm thuật toán sai.
Đoạn trace sau in cả ba trạng thái để quan sát rõ thứ tự.

In [ ]:
def trace_rows_lifecycle118(n):
    rows=[]
    for length in range(1,n+1):
        print('trước:',rows)
        row=[1]*length
        if rows:
            prev=rows[-1]
            for j in range(1,length-1):row[j]=prev[j-1]+prev[j]
        print('hàng vừa xây:',row)
        rows.append(row)
        print('sau:',rows)
    return rows
assert trace_rows_lifecycle118(4)==solve(4)

In [ ]:
def trace_pascal(n):
    rows=[]
    for length in range(1,n+1):
        new=[1]*length
        if rows:
            p=rows[-1]
            for j in range(1,length-1):
                print(f'hàng {length}, j={j}: {p[j-1]}+{p[j]}={p[j-1]+p[j]}')
                new[j]=p[j-1]+p[j]
        rows.append(new)
        print('=> hàng',length,':',new)
    return rows
trace_pascal(5)

### 7.3. Tính đối xứng: vì sao hàng Pascal đọc giống nhau từ hai phía?

Mỗi hàng Pascal đối xứng: `C(n,k) = C(n,n-k)`.
Có thể hiểu bằng cách chọn k người từ n người tương đương chọn n−k người **không được chọn**.
Cũng có thể chứng minh quy nạp từ quy tắc “ô mới là tổng hai ô phía trên”.

Đây là thuộc tính để **kiểm thử** thuật toán:
`row == row[::-1]`. Trong Python, `[::-1]` là slicing với bước `-1`, đảo list.
Lưu ý đối xứng chỉ là điều kiện cần; không đủ để kết luận hàng đúng (ví dụ `[1,0,1]` cũng đối xứng).
Cần kết hợp thêm độ dài hàng và quy tắc cộng các ô giữa.

In [ ]:
for r,row in enumerate(solve(8)):
    print('hàng',r,'đối xứng?',row==row[::-1])
    assert row==row[::-1]
assert [1,0,1]==[1,0,1][::-1]  # đối xứng nhưng KHÔNG phải hàng 2 Pascal

### 7.4. Tổng các ô của hàng r bằng `2^r` — một phép thử mạnh

Ví dụ hàng 4 `[1,4,6,4,1]` có tổng 16 = 2⁴.
Lý do: mỗi vị trí của hàng trước đóng góp vào hai vị trí ở hàng mới;
tổng tất cả phần tử vì vậy **gấp đôi** mỗi khi xuống một tầng.
Hàng 0 có tổng 1; sau r lượt gấp đôi có tổng `2**r`.

Công thức này kiểm tra được nhiều hàng nhanh mà không cần biết từng ô kỳ vọng.
Nhưng nó vẫn chưa đủ một mình để đảm bảo đúng: cần kiểm tra cả đối xứng, độ dài và
liên hệ với hàng trước. Đọc số mũ `**` trong Python là phép lũy thừa, không phải nhân hai dấu sao vô nghĩa.

In [ ]:
for r,row in enumerate(solve(9)):
    s=sum(row);expected=2**r
    print('r=',r,'sum=',s,'2**r=',expected)
    assert s==expected
assert sum([1,4,6,4,1])==16

## 8. Vì sao thuật toán đúng? — lập luận bằng bất biến

Hai đầu hàng luôn bằng 1. Theo định nghĩa tam giác Pascal, phần tử trong có tọa độ j được tính từ `prev[j-1]` và `prev[j]`; khi đã xây đúng hàng trước, vòng lặp sẽ xây đúng hàng hiện tại. Quy nạp theo số hàng.

**Hai câu cần trả lời:** (1) Sau mỗi bước, điều gì vẫn đúng? (2) Khi dừng, vì sao giá trị trả về chính là đáp án theo yêu cầu?

## 9. Code nộp LeetCode (Python 3)

**Chỉ lấy cell bên dưới để nộp.** Phần dựng input và `print()` ở các cell khác chỉ phục vụ học tập.

In [ ]:
class Solution:
    def generate(self, numRows: int):
        rows=[]
        for length in range(1,numRows+1):
            row=[1]*length
            if rows:
                prev=rows[-1]
                for j in range(1,length-1):
                    row[j]=prev[j-1]+prev[j]
            rows.append(row)
        return rows

## 10. Kiểm thử đúng đầu ra, trường hợp biên và lỗi dễ bỏ sót

Các lệnh `assert` sẽ dừng với thông báo lỗi nếu kết quả **khác** giá trị mong đợi. Tập test bao gồm ví dụ dễ hiểu, input nhỏ nhất và tình huống dễ nhầm. Đừng chỉ kiểm tra một ví dụ.

In [ ]:
for n,expect in [(1,[[1]]),(2,[[1],[1,1]]),(3,[[1],[1,1],[1,2,1]]),(5,[[1],[1,1],[1,2,1],[1,3,3,1],[1,4,6,4,1]])]:
    assert solve(n)==expect
    assert Solution().generate(n)==expect
    assert solve(n)==baseline(n)
print('PASS tam giác Pascal')

### 10.1. Bộ test không chỉ so một tam giác mẫu

Để bắt lỗi “sai mép”, “lệch một vị trí”, “ghi đè hàng cũ” và “thiếu hàng cuối”:
- Chạy n=1 và n=2: hai trường hợp không có ô giữa.
- Chạy n=3, n=5, n=10: kiểm tra truy hồi và độ dài hàng.
- So với `baseline` dùng tổ hợp (một nguồn tính độc lập).
- Với mọi hàng, xác nhận đối xứng và tổng bằng 2^r.

Đây là *property-based test* (kiểm thử theo tính chất), không phụ thuộc một list expected cố định.
Với n lớn hơn, số nguyên vẫn chính xác vì Python dùng integer có độ dài linh hoạt.

In [ ]:
for n in range(1,31):
    result=solve(n)
    assert result==baseline(n)
    assert len(result)==n
    for r,row in enumerate(result):
        assert len(row)==r+1
        assert row[0]==row[-1]==1
        assert row==row[::-1]
        assert sum(row)==2**r
print('PASS kiểm thử toàn diện n=1..30')

### 10.2. Kiểm tra mỗi hàng là một list độc lập

Một test quan trọng không chỉ nhìn **giá trị**: phải kiểm tra **tham chiếu**.
Nếu hai hàng khác nhau cùng chỉ đến một đối tượng list, sửa một hàng sẽ làm hỏng hàng kia.
Thí nghiệm test dùng `is not` (khác đối tượng) và sao chép kết quả trước khi sửa một hàng,
nhằm xác nhận lời giải không chia sẻ cùng list giữa các hàng.

Đây là kinh nghiệm chung khi dùng list lồng nhau cho ma trận DP, bảng game, đường đi,
hay tập các kết quả trung gian. Đừng chỉ tin test output ở thời điểm chưa sửa dữ liệu.

In [ ]:
triangle_independence118=solve(6)
assert all(triangle_independence118[i] is not triangle_independence118[j]
           for i in range(6) for j in range(i+1,6))
before118=[r.copy() for r in triangle_independence118]
triangle_independence118[-1][0]=777
assert triangle_independence118[:-1]==before118[:-1]
print('PASS mọi hàng lưu trong danh sách là đối tượng độc lập')

## 11. Phân tích Time/Space Complexity

Số phần tử đầu ra `1+2+...+n = n(n+1)/2`: **Time O(n²)** và **Space O(n²)** cho danh sách kết quả; bộ nhớ tạm O(n) cho mỗi hàng.

Nhớ phân biệt **số phần tử đầu vào** với **độ cao cây**, **số bit** hoặc **số chữ số**. Nếu thuật toán dùng recursion, bộ nhớ ngăn xếp lời gọi cũng cần được tính.

## 12. Các lỗi người mới hay mắc

- ❌ Dùng `row=[0]*length` rồi quên hai phần tử 1 ở đầu/cuối.
- ❌ `prev[j+1]` gây lệch chỉ số.
- ❌ `rows.append(row)` trước khi điền các ô giữa; dễ khiến dùng chính hàng mới.

Khi debug: cố tạo input nhỏ nhất gây sai, trace biến ở **ngay trước lệnh `if`/`return`**, rồi sửa nguyên nhân thay vì thêm điều kiện tùy tiện.

### 12.1. Code sai: `rows.append(row)` trước khi tính ô giữa

Nếu thêm `row` vào `rows` quá sớm rồi lấy `prev=rows[-1]`, biến prev trỏ vào
**hàng mới** chứ không phải hàng trước. Khi đọc `prev[j-1]+prev[j]` bạn đang
lấy hai số 1 ban đầu của hàng hiện tại, tạo kết quả sai.

Thứ tự đúng:
(1) `prev=rows[-1]` khi rows chưa chứa hàng mới;
(2) tính các ô của row;
(3) `rows.append(row)`.
Hãy nhìn ví dụ n=4, nơi lỗi chưa chắc lộ ở hai hàng đầu nhưng chắc chắn sai ở hàng 3.

In [ ]:
def wrong_append_early118(n):
    rows=[]
    for length in range(1,n+1):
        row=[1]*length
        rows.append(row)       # SAI: thêm trước khi lấy prev
        if length>1:
            prev=rows[-1]      # là row, không phải hàng trước
            for j in range(1,length-1):row[j]=prev[j-1]+prev[j]
    return rows
print('sai:',wrong_append_early118(4))
print('đúng:',solve(4))
assert wrong_append_early118(4)!=solve(4)

### 12.2. Code sai: phạm vi `j` đi qua mép phải

Nếu vòng lặp là `for j in range(1,length)` thì j có thể bằng length−1,
là **vị trí mép phải**. Khi ấy `prev[j]` vượt chỉ số cuối hàng trước.
Đề yêu cầu giữ hai mép bằng 1, chỉ các ô trong mới cộng từ hàng trước.

Với `length=5`, hàng trước có 4 phần tử chỉ số 0..3,
và vị trí cần cập nhật là 1,2,3; j=4 sẽ truy cập `prev[4]` và gây `IndexError`.
Hãy in các chỉ số thay vì chạy một hàm cố tình lỗi liên tục.
Lỗi tương tự có thể xuất hiện trong KNN, sliding window và các bài so phần tử kế bên.

In [ ]:
length118=5
prev118=[1,3,3,1]
correct_indexes=list(range(1,length118-1))
wrong_indexes=list(range(1,length118))
print('đúng:',correct_indexes,'sai:',wrong_indexes)
print('cột cuối hợp lệ của prev:',len(prev118)-1)
assert correct_indexes==[1,2,3]
assert wrong_indexes[-1]==len(prev118)  # index out of bounds

## 13. Nhận diện dạng bài và vận dụng

Khi một ô **phụ thuộc kết quả liền trước** và các bài toán con lặp lại, xem xét DP: xây từ trường hợp nhỏ, lưu đáp án trung gian.

**Chuyển giao:** hãy tìm một bài khác có cùng dấu hiệu đầu vào và tập trung vào lý do giữ bất biến, không học thuộc tên biến.

## 14. Biến thể có ý nghĩa

Tính tổng từng hàng: hàng `r` có tổng `2^r` (với `r` bắt đầu từ 0). Kiểm tra quy luật bằng code.

Biến thể này dùng lại kỹ thuật vừa học, nhưng có một thay đổi nhỏ giúp kiểm tra bạn đã hiểu hay chưa.

In [ ]:
for i,row in enumerate(solve(6)):
    print(f'hàng {i}: sum={sum(row)}, 2^{i}={2**i}')

### 14.1. Từ toàn bộ tam giác sang một hàng riêng lẻ

Bài #118 xuất **toàn bộ** bảng hai chiều. Nếu chỉ cần hàng k,
bạn có thể (1) sinh k+1 hàng rồi lấy `rows[-1]`, hoặc (2) tối ưu bộ nhớ để chỉ giữ một hàng.
Bài #119 kế tiếp chính là lựa chọn (2), và có một vấn đề mới:
**khi cập nhật cùng một list, phải đi từ phải sang trái** để giữ số cũ cần dùng.

Mục tiêu của bài biến thể không phải học trước toàn bộ #119,
mà nhận ra dữ liệu nào đầu ra thật sự cần giữ lại.
Nếu đề bắt trả tất cả hàng thì không thể giảm riêng bộ nhớ *đầu ra* từ O(n²) xuống O(n).

In [ ]:
for target_row in (0,1,4,7):
    one_row=solve(target_row+1)[-1]
    print('hàng',target_row,':',one_row)
    assert one_row==[comb(target_row,j) for j in range(target_row+1)]

### 14.2. Tổ hợp và truy hồi là cùng một quy luật

Tại hàng n và cột k, `C(n,k)=C(n-1,k-1)+C(n-1,k)`.
Giải thích bằng cách xét một phần tử đặc biệt X trong tập n phần tử:
- Chọn X: phải chọn k−1 phần tử từ n−1 phần tử còn lại.
- Không chọn X: phải chọn k phần tử từ n−1 phần tử còn lại.
Hai trường hợp không trùng, nên cộng số cách chọn.

Như vậy công thức Pascal không phải mẹo ghi nhớ vô cớ:
nó là cách tách một bài đếm thành hai trường hợp nhỏ hơn.
Hãy thử ba ô giữa của hàng 6 rồi đối chiếu `math.comb` và hàng Pascal được dựng bởi DP.

In [ ]:
for k in range(1,6):
    left=comb(5,k-1);right=comb(5,k)
    combined=comb(6,k)
    print('C(6,',k,') =',left,'+',right,'=',combined)
    assert left+right==combined
assert solve(7)[6]==[comb(6,k) for k in range(7)]

## 15. Tự luyện — 5 bài tăng dần

**Mục tiêu:** tự hiểu cú pháp, trace bằng tay, viết lại thuật toán, chuyển sang biến thể, rồi thực hiện mini challenge. Tất cả ví dụ giữ nguyên đúng bản chất đề LeetCode. **Đừng xem đáp án trước.**

### Bài 1 — Dự đoán output của Python (dễ)

Cho `previous=[1,3,3,1]`. Dự đoán `previous[0]+previous[1]`, `previous[1]+previous[2]`, `previous[2]+previous[3]`.

Hãy viết output ra giấy trước khi chạy cell dưới.

In [ ]:
previous=[1,3,3,1]
print(previous[0]+previous[1])
print(previous[1]+previous[2])
print(previous[2]+previous[3])

### Bài 2 — Trace bằng tay (hiểu bản chất)

Viết tay **5 hàng đầu tiên** (tính từ hàng 0), giải thích vì sao hàng số 4 có đúng 5 phần tử.

Tự viết bảng gồm **trạng thái trước**, phép kiểm tra và **trạng thái sau**; không chỉ ghi kết quả cuối.

### Bài 3 — TODO: Tự viết lời giải chính

Viết `exercise_generate_pascal(numRows)` sinh danh sách của `numRows` hàng; `numRows=0` trả list rỗng.

Các hàm bài tập được định nghĩa độc lập và **không được gọi** khi chưa hoàn thiện.

In [ ]:
def exercise_generate_pascal(numRows):
    # TODO: từ hàng [1], xây hàng mới dựa trên hàng trước.
    raise NotImplementedError

### Bài 4 — TODO: Chuyển giao sang biến thể

Viết `exercise_check_pascal_symmetry(rows)` nhận **một tam giác bất kỳ** và trả `True` nếu mọi hàng đều đối xứng trái–phải.

Hãy thử input nhỏ nhất và một phản ví dụ. Dự đoán output trước khi gọi.

In [ ]:
def exercise_check_pascal_symmetry(rows):
    # TODO: so mỗi row với list đảo ngược row[::-1].
    raise NotImplementedError

### Bài 5 — TODO: Mini challenge

Mini challenge: `exercise_pascal_nth_row(n)` tự tính trực tiếp hàng `n` bằng công thức C(n,k), không phải xây cả tam giác. Công thức lặp: `c_next = c * (n-k) // (k+1)`.

Hãy nêu độ phức tạp của hàm và thử tối thiểu 3 ca biên.

In [ ]:
def exercise_pascal_nth_row(n):
    # TODO: bắt đầu c=1; với k từ 0..n, append c rồi cập nhật.
    raise NotImplementedError

### Bài 6 — TODO: tự kiểm tra tam giác Pascal đã sinh có hợp lệ không

Viết `exercise_validate_pascal(rows)` trả `True` khi `rows` chính xác là một tam giác Pascal
với số hàng bằng `len(rows)`. Với `[]` (mở rộng ngoài phạm vi n≥1 của đề gốc), trả `True`.

Các điều kiện cần kiểm tra: hàng r phải dài r+1, hai đầu bằng 1,
mọi ô giữa bằng tổng hai ô cùng vị trí k−1 và k của hàng trước.
Hàm phải phát hiện được tam giác giả `[ [1], [1,1], [1,0,1] ]` dù có đối xứng.

Gợi ý 1: duyệt từng hàng bằng `enumerate`.
Gợi ý 2: bắt lỗi hình dạng trước khi truy cập chỉ số.
Gợi ý 3: với r≥2, duyệt k từ 1 đến r−1 để xác minh công thức cộng.

In [ ]:
def exercise_validate_pascal(rows):
    # TODO: phát hiện dữ liệu giả mạo, sai mép, sai độ dài, sai ô giữa.
    # Không gọi solve hoặc baseline; hãy tự kiểm tra bằng quy tắc địa phương.
    raise NotImplementedError

# Tự làm mới bỏ comment:
# assert exercise_validate_pascal([[1],[1,1],[1,2,1]])
# assert not exercise_validate_pascal([[1],[1,1],[1,0,1]])

## 16. Gợi ý mở dần — đừng đọc cùng lúc

<details><summary><b>Cấp 1 · Trực giác</b></summary>

Các phần tử ở hai biên luôn bằng 1; phần tử giữa bằng tổng hai ô **phía trên**.

</details>

<details><summary><b>Cấp 2 · Dữ liệu và cú pháp</b></summary>

Hàng i có i+1 số; công thức trực tiếp dùng phép chia nguyên `//` để giữ số nguyên.

</details>

<details><summary><b>Cấp 3 · Gần-code</b></summary>

Sinh hàng mới: `[1] + [prev[j-1]+prev[j] for j in range(1,i)] + [1]` với i≥1.

</details>

## 17. Đáp án tham khảo — chỉ đọc sau khi tự làm

### Bài 1 — Output

Kết quả: `4`, `6`, `4`. Khi thêm hai đầu bằng 1 ta có `[1,4,6,4,1]`.

### Bài 2 — Trace tay

Các hàng: `[1]`, `[1,1]`, `[1,2,1]`, `[1,3,3,1]`, `[1,4,6,4,1]`. Hàng index `i` có `i+1` phần tử vì thêm hai đầu và chèn các tổng cặp cạnh nhau.

### Đáp án bài 3 — Viết lại giải pháp chính

Đọc rõ base case và ý nghĩa giá trị trả về. Đáp án độc lập, không dựa vào `solve` phía trên.

In [ ]:
def answer_generate_pascal(numRows):
    rows=[]
    for i in range(numRows):
        if i==0: rows.append([1]);continue
        prev=rows[-1]
        row=[1]+[prev[j-1]+prev[j] for j in range(1,i)]+[1]
        rows.append(row)
    return rows

### Đáp án bài 4 — Biến thể

So sánh **trạng thái cần mang theo** giữa bài chính và biến thể.

In [ ]:
def answer_check_pascal_symmetry(rows):
    for row in rows:
        if row!=row[::-1]: return False
    return True

### Đáp án bài 5 — Mini challenge

Kiểm tra kỹ trường hợp rỗng, đầu/cuối, và các điều kiện dừng.

In [ ]:
def answer_pascal_nth_row(n):
    row=[];c=1
    for k in range(n+1):
        row.append(c)
        if k<n: c=c*(n-k)//(k+1)
    return row

### Đáp án bài 6 — kiểm tra từ cấu trúc đến giá trị

Đầu tiên, kiểm tra số phần tử của hàng r có phải r+1 không.
Sau đó kiểm tra cả hai biên bằng 1. Cuối cùng, so mọi ô giữa với hai ô hàng trước.
Việc kiểm tra theo thứ tự này rất quan trọng: nếu hàng có độ dài sai mà vẫn truy cập
`prev[k]`, hàm kiểm tra có thể tự ném `IndexError` thay vì trả `False`.

Hàm có O(n²) thời gian (xét tổng cộng số ô tam giác) và O(1) bộ nhớ phụ
không tính chính input. Đây là dạng **validate cấu trúc DP** có thể dùng lại cho các bảng khác.

In [ ]:
def answer_validate_pascal(rows):
    for r,row in enumerate(rows):
        if len(row)!=r+1:return False
        if row[0]!=1 or row[-1]!=1:return False
        if r>=2:
            prev=rows[r-1]
            for k in range(1,r):
                if row[k]!=prev[k-1]+prev[k]:return False
    return True
for n in range(0,16):
    assert answer_validate_pascal(solve(n))
assert not answer_validate_pascal([[1],[1,1],[1,0,1]])
assert not answer_validate_pascal([[1],[1,1,1]])
assert not answer_validate_pascal([[1],[0,1]])
print('PASS bài 6: validate tam giác Pascal')

### Kiểm thử 3 lời giải bài tập

Các assert sau **chỉ kiểm tra đáp án mẫu**, không chạy các hàm TODO của bạn. Khi hoàn thiện TODO, bạn hãy tự thay tên hàm trong test để đánh giá.

In [ ]:
rows=answer_generate_pascal(5)
assert rows==[[1],[1,1],[1,2,1],[1,3,3,1],[1,4,6,4,1]]
assert answer_generate_pascal(0)==[]
assert answer_check_pascal_symmetry(rows)
assert not answer_check_pascal_symmetry([[1],[1,2]])
for k in range(11): assert answer_pascal_nth_row(k)==answer_generate_pascal(k+1)[-1]
print("PASS: đáp án 3 bài coding của phần luyện tập")

## 18. Tóm tắt và checklist submit

**Nhớ nhanh:** Đây là **Dynamic Programming**: một hàng được xây từ kết quả **hàng trước**. Ta không cần tính lại giai thừa.

- [ ] Tôi tự nói được input, output và điều kiện biên.
- [ ] Tôi hiểu từng cú pháp Python trong thuật toán.
- [ ] Tôi có thể tự vẽ bảng trace ít nhất một ví dụ.
- [ ] Tôi giải thích được **tại sao đúng** mà không dựa vào test.
- [ ] Tôi phân biệt được Time/Space của baseline và lời giải chính.
- [ ] Tôi copy đúng phần class LeetCode, không copy hàm dựng cây/list hay `print()`.
- [ ] Tôi tự làm xong các TODO, rồi mới đọc đáp án.

**Bài tiếp theo:** duy trì thói quen trace từ trạng thái nhỏ và chỉ tối ưu khi đã hiểu bản chất.